# Recs 004: Offline eval — same-user held-out likes (proxy task)

**Sections:** **§1** setup · **§2** sample users · **§3** raw / structured / **random** / **popularity** baselines · **§4** **multi-review** (mean vs concat).

**Goal:** Compare **raw** vs **structured** query embedding on a concrete relevance definition: other games the **same** user thumbs-up reviewed (`recommended == 1`), excluding the query game. See **`docs/recommender_transition_plan.md`** → *Offline proxy task: other games the same user liked*.

**Requires:** [`recs_002`](./recs_002_embed_game_profiles.ipynb) artifacts (`game_profile_embeddings.npz`, index Parquet, `meta.json`). Uses the **same TF Hub** model as `recs_003`.

**Split choice:** Query reviews come from the **validation** split (`*_val_norm.parquet`) so you do not burn the **test** holdout while iterating. Game vectors are still built from **train** (`recs_001` / `recs_002`). If val is missing, the notebook falls back to **train** and prints a warning (query text may overlap game-profile pools).

**Caveats:** Users with only one indexed thumbs-up review are skipped. Possible **franchise correlation** among positives. **Recall@K** is normalized by \|positives\| so users with many likes have harder scores.

**Empirical note (val proxy, rules-based `extract_preferences`):** **raw** embedding usually **beats** **structured** on Hit@K / Recall@K / MRR here. Treat **raw** as the **default** query for USE + this eval until structured improves; keep structured as an **ablation** (see `docs/recommender_transition_plan.md`).

**On a small catalog, our same-user proxy correlates with popularity; we report a popularity baseline and treat beating it as a separate milestone — raw text similarity is not yet personalized enough.**



## 1) Paths, game matrix, embedder


In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# Repo root (same pattern as recs_003)
def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root (no pyproject.toml). cwd={here}")


REPO_ROOT = _repo_root()
PROCESSED = REPO_ROOT / "data" / "processed"
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "recs"
NPZ_PATH = ARTIFACT_DIR / "game_profile_embeddings.npz"
INDEX_PATH = ARTIFACT_DIR / "game_profile_embedding_index.parquet"
META_PATH = ARTIFACT_DIR / "game_profile_embedding_meta.json"

VAL_PARQUET = PROCESSED / "steam_reviews_cleaned_english_val_norm.parquet"
TRAIN_PARQUET = PROCESSED / "steam_reviews_cleaned_english_train_norm.parquet"
# Val for iterative eval; reserve test_norm for a final holdout run only.
EVAL_PARQUET = VAL_PARQUET if VAL_PARQUET.is_file() else TRAIN_PARQUET
EVAL_SPLIT_NAME = "val" if EVAL_PARQUET == VAL_PARQUET else "train"

for p in (NPZ_PATH, INDEX_PATH, META_PATH):
    if not p.is_file():
        raise FileNotFoundError(f"Run recs_002 first. Missing {p}")
if not EVAL_PARQUET.is_file():
    raise FileNotFoundError(f"Missing {EVAL_PARQUET} — run normalization pipeline (see docs/usage_pipeline.md)")
if not TRAIN_PARQUET.is_file():
    raise FileNotFoundError(f"Missing {TRAIN_PARQUET} — needed for popularity baseline")

meta = json.loads(META_PATH.read_text(encoding="utf-8"))
TFHUB_URL = meta["model_name"]
EMBED_DIM = int(meta["dim"])
MAX_CHARS = meta.get("max_chars_per_review")

print("Eval split:", EVAL_SPLIT_NAME, "→", EVAL_PARQUET.name)
print("TF Hub:", TFHUB_URL, "| dim:", EMBED_DIM)


Eval split: val → steam_reviews_cleaned_english_val_norm.parquet
TF Hub: https://tfhub.dev/google/universal-sentence-encoder/4 | dim: 512


In [2]:
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
import tensorflow_hub as hub

for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

embed_fn = hub.load(TFHUB_URL)


I0000 00:00:1775914375.003147   28760 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775914377.403695   28760 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [3]:
z = np.load(NPZ_PATH)
X = np.asarray(z["embeddings"], dtype=np.float32)
app_ids_X = np.asarray(z["app_id"], dtype=np.int64)
z.close()

idx_df = pd.read_parquet(INDEX_PATH)
if len(idx_df) != X.shape[0] or not np.array_equal(idx_df["app_id"].to_numpy(), app_ids_X):
    raise ValueError("Index / npz app_id alignment")

# row i <-> app_ids_X[i]
app_to_row = {int(a): i for i, a in enumerate(app_ids_X)}
indexed_apps = set(app_to_row.keys())
n_games = X.shape[0]
print("X:", X.shape, "unique games in index:", n_games)


X: (315, 512) unique games in index: 315


In [4]:
def l2_normalize(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32).ravel()
    nrm = np.linalg.norm(v)
    if nrm <= 1e-12:
        return v
    return (v / nrm).astype(np.float32)


def embed_text(text: str) -> np.ndarray:
    t = (text or "").strip()
    if MAX_CHARS is not None:
        t = t[: int(MAX_CHARS)]
    out = embed_fn([t])
    return l2_normalize(out)


def scores_excluding_query(q: np.ndarray, query_app_id: int) -> np.ndarray:
    s = (X @ q).astype(np.float32)
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return s


def rank_app_ids(s: np.ndarray) -> np.ndarray:
    """Indices into X rows, highest score first."""
    return np.argsort(-s)


def recall_at_k(ranked_rows: np.ndarray, positives: set[int], k: int) -> float:
    top = set(int(app_ids_X[i]) for i in ranked_rows[:k])
    if not positives:
        return float("nan")
    return len(top & positives) / len(positives)


def hit_rate_at_k(ranked_rows: np.ndarray, positives: set[int], k: int) -> float:
    top = set(int(app_ids_X[i]) for i in ranked_rows[:k])
    return 1.0 if (top & positives) else 0.0


def mrr(ranked_rows: np.ndarray, positives: set[int]) -> float:
    for rank, i in enumerate(ranked_rows.tolist(), start=1):
        if int(app_ids_X[i]) in positives:
            return 1.0 / rank
    return 0.0


## 2) Sample eval rows (multi-review users)

Tune **`RNG_SEED`**, **`MAX_USERS`**, **`MIN_REVIEW_CHARS`**. Each sampled user contributes **one** query: we pick a random thumbs-up review as the query and treat their **other** indexed thumbs-up `app_id`s as positives. We also stash **`support_texts`** (one random review per other liked game) for **§4**.


In [5]:
from IPython.display import display

from steam_review_ml.recommender import build_embedding_input, extract_preferences

USER_COL = "author.steamid"
MIN_REVIEW_CHARS = 30
RNG_SEED = 42
MAX_USERS = 5000  # cap for notebook runtime; raise for stabler metrics

rng = np.random.default_rng(RNG_SEED)

usecols = [USER_COL, "app_id", "review", "recommended", "review_id"]
df = pd.read_parquet(EVAL_PARQUET, columns=usecols)
df = df.loc[df["recommended"] == 1].copy()
df["review"] = df["review"].fillna("").astype(str)
df = df[df["review"].str.len() >= MIN_REVIEW_CHARS]
df = df[df["app_id"].isin(indexed_apps)]

# One row per (user, app): keep a random review if duplicates exist
df["_k"] = rng.random(len(df))
df = df.sort_values("_k").drop_duplicates(subset=[USER_COL, "app_id"], keep="first").drop(columns=["_k"])

uc = df.groupby(USER_COL)["app_id"].nunique()
multi = uc[uc >= 2].index
multi = pd.Index(rng.permutation(multi.values)[: min(len(multi), MAX_USERS)])

examples: list[dict] = []
for uid in multi:
    sub = df[df[USER_COL] == uid]
    apps = sub["app_id"].unique().tolist()
    q_app = int(rng.choice(apps))
    rows_q = sub[sub["app_id"] == q_app]
    row = rows_q.iloc[rng.integers(0, len(rows_q))]
    positives = {int(a) for a in apps if int(a) != q_app}
    support_texts: list[str] = []
    for oa in apps:
        if int(oa) == q_app:
            continue
        rows_oa = sub[sub["app_id"] == oa]
        if len(rows_oa):
            support_texts.append(str(rows_oa.iloc[rng.integers(0, len(rows_oa))]["review"]))
    rng.shuffle(support_texts)
    examples.append(
        {
            "steamid": uid,
            "query_app_id": q_app,
            "query_text": str(row["review"]),
            "positives": positives,
            "n_pos": len(positives),
            "support_texts": support_texts,
        }
    )

print(f"Examples: {len(examples)} users (multi-review, thumbs-up, in index)")
print("Positives per example: min", min(e["n_pos"] for e in examples), "max", max(e["n_pos"] for e in examples))


Examples: 5000 users (multi-review, thumbs-up, in index)
Positives per example: min 1 max 8


In [9]:
support_texts

['Mac Steam/Linux Support? Thanks!',
 "If you don't play this, you don't deserve to live"]

## 3) Run retrieval: raw vs structured vs baselines

Metrics: **HitRate@K**, **Recall@K**, **MRR**.

- **random** — uniform random scores per game (query `app_id` masked).
- **popularity_train** — rank by **train** thumbs-up count per indexed `app_id` (query masked).


In [6]:
KS = (5, 10, 20)

# Popularity: train thumbs-up counts aligned to X rows
_train_use = ["app_id", "recommended"]
_df_tr = pd.read_parquet(TRAIN_PARQUET, columns=_train_use)
_df_tr = _df_tr.loc[_df_tr["recommended"] == 1]
_vc = _df_tr.groupby("app_id").size()
pop_row = np.asarray([float(_vc.get(int(a), 0)) for a in app_ids_X], dtype=np.float32)
pop_row = np.maximum(pop_row, 1e-6)


def summarize(name: str, agg: dict) -> pd.Series:
    out = {}
    for k, v in agg.items():
        a = np.asarray(v, dtype=np.float64)
        out[k] = float(np.nanmean(a)) if k.startswith("recall") or k == "mrr" else float(a.mean())
    return pd.Series(out, name=name)


def eval_loop(q_for_ex) -> dict:
    agg = {f"hit@{k}": [] for k in KS} | {f"recall@{k}": [] for k in KS} | {"mrr": []}
    for ex in examples:
        q = q_for_ex(ex)
        s = scores_excluding_query(q, ex["query_app_id"])
        order = rank_app_ids(s)
        pos = ex["positives"]
        for k in KS:
            agg[f"hit@{k}"].append(hit_rate_at_k(order, pos, k))
            agg[f"recall@{k}"].append(recall_at_k(order, pos, k))
        agg["mrr"].append(mrr(order, pos))
    return agg


def eval_random_baseline() -> dict:
    agg = {f"hit@{k}": [] for k in KS} | {f"recall@{k}": [] for k in KS} | {"mrr": []}
    for ex in examples:
        s = rng.random(n_games).astype(np.float32)
        row = app_to_row.get(int(ex["query_app_id"]))
        if row is not None:
            s[row] = -np.inf
        order = np.argsort(-s)
        pos = ex["positives"]
        for k in KS:
            agg[f"hit@{k}"].append(hit_rate_at_k(order, pos, k))
            agg[f"recall@{k}"].append(recall_at_k(order, pos, k))
        agg["mrr"].append(mrr(order, pos))
    return agg


def eval_popularity_baseline() -> dict:
    agg = {f"hit@{k}": [] for k in KS} | {f"recall@{k}": [] for k in KS} | {"mrr": []}
    for ex in examples:
        s = pop_row.copy()
        row = app_to_row.get(int(ex["query_app_id"]))
        if row is not None:
            s[row] = -np.inf
        order = np.argsort(-s)
        pos = ex["positives"]
        for k in KS:
            agg[f"hit@{k}"].append(hit_rate_at_k(order, pos, k))
            agg[f"recall@{k}"].append(recall_at_k(order, pos, k))
        agg["mrr"].append(mrr(order, pos))
    return agg


agg_rand = eval_random_baseline()
agg_pop = eval_popularity_baseline()
agg_raw = eval_loop(lambda ex: embed_text(ex["query_text"]))
agg_struct = eval_loop(
    lambda ex: embed_text(build_embedding_input(extract_preferences(ex["query_text"]), ex["query_text"]))
)

summary = pd.DataFrame(
    [
        summarize("random", agg_rand),
        summarize("popularity_train", agg_pop),
        summarize("raw", agg_raw),
        summarize("structured", agg_struct),
    ]
)
display(summary.T)

print("Split:", EVAL_SPLIT_NAME, "| n_examples:", len(examples), "| n_games:", n_games)
if EVAL_SPLIT_NAME == "train":
    print("Note: queries are from TRAIN — query text may appear inside game-profile pools. Prefer val_norm when available.")
elif EVAL_SPLIT_NAME == "val":
    print("Using validation queries; keep test_norm untouched for a final evaluation pass.")


E0000 00:00:1775914395.784252   28760 util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


,random,popularity_train,raw,structured
hit@5,0.018400,0.122000,0.049600,0.025800
hit@10,0.038400,0.218200,0.084400,0.048400
hit@20,0.078600,0.377000,0.154600,0.098000
recall@5,0.014244,0.103692,0.040695,0.020602
recall@10,0.029872,0.186048,0.069535,0.037669
recall@20,0.061401,0.327236,0.129180,0.079117
mrr,0.023173,0.100967,0.043749,0.028126


Split: val | n_examples: 5000 | n_games: 315
Using validation queries; keep test_norm untouched for a final evaluation pass.


## 4) Multi-review query (mean vs concat-cap)

Pool **several validation reviews** from the **same user**: **mean embedding** (L2-normalize after mean) vs **one embed** on **concatenated** text (capped).

**Leakage note:** `support_texts` can include reviews for **other games this user liked** (positives). Treat this as an **optimistic / user-profile** check (“does more text help?”), not a cold query-only setup. For strict leave-one-out, exclude held-out positives from the pool.

Tune **`MULTI_MAX_REVIEWS`** and **`MULTI_CONCAT_CHARS`** below.


In [7]:
MULTI_MAX_REVIEWS = 5
MULTI_CONCAT_CHARS = 2000


def q_multi_mean(ex: dict) -> np.ndarray:
    texts = [ex["query_text"].strip()]
    for t in ex.get("support_texts", [])[: max(0, MULTI_MAX_REVIEWS - 1)]:
        if t and t.strip():
            texts.append(t.strip())
    if len(texts) == 1:
        return embed_text(texts[0])
    vecs = np.stack([embed_text(t) for t in texts], axis=0).astype(np.float32)
    return l2_normalize(vecs.mean(axis=0))


def q_multi_concat(ex: dict) -> np.ndarray:
    parts = [ex["query_text"].strip()]
    for t in ex.get("support_texts", []):
        if len(parts) >= MULTI_MAX_REVIEWS:
            break
        if t and t.strip():
            parts.append(t.strip())
        if sum(len(p) + 2 for p in parts) >= MULTI_CONCAT_CHARS:
            break
    blob = " \n\n ".join(parts)[:MULTI_CONCAT_CHARS]
    return embed_text(blob)


agg_m_mean = eval_loop(q_multi_mean)
agg_m_cat = eval_loop(q_multi_concat)

summary_multi = pd.DataFrame(
    [
        summarize("raw_single_review", agg_raw),
        summarize("multi_mean", agg_m_mean),
        summarize("multi_concat", agg_m_cat),
    ]
)
display(summary_multi.T)


,raw_single_review,multi_mean,multi_concat
hit@5,0.049600,0.307600,0.290200
hit@10,0.084400,0.403400,0.378600
hit@20,0.154600,0.524200,0.482000
recall@5,0.040695,0.269438,0.251328
recall@10,0.069535,0.358012,0.329200
recall@20,0.129180,0.469502,0.425460
mrr,0.043749,0.226670,0.224707
